# Brian2: Spiking Neural Networks

Simulate Hodgkin-Huxley and leaky integrate-and-fire neurons, balanced E/I networks, and STDP.

## Prerequisites
- Basic understanding of differential equations and neuronal membrane dynamics
- Familiarity with Python and matplotlib
- Conceptual knowledge of action potentials, synaptic transmission, and neural coding

## Setup
Run the cell below to install dependencies and download data.

In [ ]:
!pip install brian2 matplotlib

In [ ]:
from brian2 import *
import matplotlib.pyplot as plt

# --- 1. Leaky Integrate-and-Fire network (E/I balanced) ---
start_scope()
N = 1000
tau = 10*ms
v_thresh = -50*mV
v_rest = -70*mV
# I_ext must be declared as a state variable in the equations string
eqs = '''
    dv/dt = (v_rest - v + I_ext) / tau : volt
    I_ext : volt
'''
G = NeuronGroup(N, eqs, threshold='v > v_thresh', reset='v = v_rest', method='euler')
G.v = 'v_rest + rand() * 20*mV'
G.I_ext = 20*mV
S = Synapses(G, G, on_pre='v_post += 0.5*mV')
S.connect(p=0.1)
M = SpikeMonitor(G)
run(200*ms)
plt.figure(figsize=(12,4))
plt.plot(M.t/ms, M.i, '.k', markersize=1)
plt.xlabel('Time (ms)'); plt.ylabel('Neuron index')
plt.title('Raster plot \u2014 LIF network (N=1000)')
plt.tight_layout(); plt.show()
print(f'Mean firing rate: {M.num_spikes / (N * 200e-3):.1f} Hz')

In [ ]:
# --- 2. STDP (Spike-Timing Dependent Plasticity) ---
start_scope()
tau_pre = tau_post = 20*ms
A_pre = 0.01; A_post = -A_pre * 1.05

# NeuronGroup equations must NOT have (event-driven) flag \u2014 that's for Synapses only
eqs_neuron = 'dv/dt = -v/tau : volt'

# apre/apost belong in the Synapses model string with (event-driven) flag
eqs_syn = '''w : 1
    dapre/dt = -apre/tau_pre : 1 (event-driven)
    dapost/dt = -apost/tau_post : 1 (event-driven)
'''
pre_code  = 'v_post += 0.5*mV; apre += A_pre; w = clip(w + apost, 0, 1)'
post_code = 'apost += A_post; w = clip(w + apre, 0, 1)'

G = NeuronGroup(2, eqs_neuron, threshold='v > -50*mV', reset='v = -70*mV')
G.v = -70*mV
S = Synapses(G, G, eqs_syn, on_pre=pre_code, on_post=post_code)
S.connect(i=0, j=1); S.w = 0.5
Ws = StateMonitor(S, 'w', record=True)

# Drive neuron 0 with Poisson input
PG = PoissonGroup(1, rates=80*Hz)
PS = Synapses(PG, G[0:1], on_pre='v_post += 3*mV')
PS.connect()

run(1*second)
plt.figure(figsize=(10, 4))
plt.plot(Ws.t/ms, Ws.w[0])
plt.xlabel('Time (ms)'); plt.ylabel('Synaptic weight')
plt.title('STDP weight evolution'); plt.tight_layout(); plt.show()

## References

- Stimberg, M., Brette, R., & Goodman, D. F. M. (2019). Brian 2, an intuitive and efficient neural simulator. *eLife*, 8, e47314. https://doi.org/10.7554/eLife.47314
- Hodgkin, A. L., & Huxley, A. F. (1952). A quantitative description of membrane current and its application to conduction and excitation in nerve. *The Journal of Physiology*, 117(4), 500\u2013544.
- Bi, G., & Poo, M. (1998). Synaptic modifications in cultured hippocampal neurons: dependence on spike timing, synaptic strength, and postsynaptic cell type. *Journal of Neuroscience*, 18(24), 10464\u201310472.
- Song, S., Miller, K. D., & Abbott, L. F. (2000). Competitive Hebbian learning through spike-timing-dependent synaptic plasticity. *Nature Neuroscience*, 3(9), 919\u2013926.